## 🏨 NYC AIRBNB - Data Engineering Pipeline
## Dataset: New York City Airbnb - 2019

In [41]:
import pyarrow
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Extract - Leitura dos Dados

In [42]:
df = pd.read_csv('../data/AB_NYC_2019.csv')
print(f"Total de registros: {len(df):,}")
print(f"Total de colunas: {len(df.columns)}")
df.head()

Total de registros: 48,895
Total de colunas: 16


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  str    
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  str    
 4   neighbourhood_group             48895 non-null  str    
 5   neighbourhood                   48895 non-null  str    
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  str    
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review                     38843 non-n

In [44]:
df.describe()

,id,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
count,4.889500e+04,4.889500e+04,48895.000000,48895.000000,48895.000000,48895.000000,48895.000000,38843.000000,48895.000000,48895.000000
mean,1.901714e+07,6.762001e+07,40.728949,-73.952170,152.720687,7.029962,23.274466,1.373221,7.143982,112.781327
std,1.098311e+07,7.861097e+07,0.054530,0.046157,240.154170,20.510550,44.550582,1.680442,32.952519,131.622289
min,2.539000e+03,2.438000e+03,40.499790,-74.244420,0.000000,1.000000,0.000000,0.010000,1.000000,0.000000
25%,9.471945e+06,7.822033e+06,40.690100,-73.983070,69.000000,1.000000,1.000000,0.190000,1.000000,0.000000
50%,1.967728e+07,3.079382e+07,40.723070,-73.955680,106.000000,3.000000,5.000000,0.720000,1.000000,45.000000
75%,2.915218e+07,1.074344e+08,40.763115,-73.936275,175.000000,5.000000,24.000000,2.020000,2.000000,227.000000
max,3.648724e+07,2.743213e+08,40.913060,-73.712990,10000.000000,1250.000000,629.000000,58.500000,327.000000,365.000000


## 2. Data Quality & Anomalias
Detectar e remover registros inválidos

In [45]:
print("--- Valores Nulos ---")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "Nenhum valor nulo encontrado!")

--- Valores Nulos ---
name                    16
host_name               21
last_review          10052
reviews_per_month    10052
dtype: int64


In [49]:
df_clean = df.copy()
df_clean.drop(['host_name', 'last_review'], axis= 1, inplace=True)


In [51]:
df_clean.head()

,id,name,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,0.10,1,0


In [52]:
df_clean.fillna({'reviews_per_month': 0}, inplace=True)
df_clean.fillna({'name': 'No Data'}, inplace=True)

,id,name,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,0.00,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,0.10,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48890,36484665,Charming one bedroom - newly renovated rowhouse,8232441,Brooklyn,Bedford-Stuyvesant,40.67853,-73.94995,Private room,70,2,0,0.00,2,9
48891,36485057,Affordable room in Bushwick/East Williamsburg,6570630,Brooklyn,Bushwick,40.70184,-73.93317,Private room,40,4,0,0.00,2,36
48892,36485431,Sunny Studio at Historical Neighborhood,23492952,Manhattan,Harlem,40.81475,-73.94867,Entire home/apt,115,10,0,0.00,1,27
48893,36485609,43rd St. Time Square-cozy single bed,30985759,Manhattan,Hell's Kitchen,40.75751,-73.99112,Shared room,55,1,0,0.00,6,2


In [53]:
print("--- Valores Nulos ---")
null_counts = df_clean.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "Nenhum valor nulo encontrado!")

--- Valores Nulos ---
Nenhum valor nulo encontrado!


In [54]:
total = len(df_clean)
# Coordenadas fora de NYC (latitude ~40.4-41.0, longitude ~-74.3 a -73.7)
invalid_coords = (
    (df_clean['latitude'] < 40.4) | (df_clean['latitude'] > 41.0) |
    (df_clean['longitude'] < -74.3) | (df_clean['longitude'] > -73.7)
)
print(f"Coordenadas airbnb fora de NYC:     {invalid_coords.sum():,} ({invalid_coords.sum()/total*100:.2f}%)")

Coordenadas airbnb fora de NYC:     0 (0.00%)


## 3. Transform - Colunas Calculadas 
## Enriquecer o dataset com métricas derivadas

#### Preço e Valor

In [55]:
# Ajuste no preço por noite
df_clean['price_per_min_night'] = df_clean['price'] * df_clean['minimum_nights']

# Preço por Review
df_clean['price_per_review'] = df_clean['price'] / (df_clean['number_of_reviews'] + 1)

# Categoria por Preço
df_clean['price_category'] = pd.cut(df_clean['price'],
                              bins=[0,50,150,300,1000],
                              labels=['cheap', 'medium', 'expensive', 'luxury'])

#### Ocupação e Demanda

In [56]:
# Review por Mês - intensidade
df_clean['demand_level'] = pd.cut(df_clean['reviews_per_month'],
                            bins=[0,1,3,10],
                            labels=['low', 'medium', 'high'])

# Atividade do host
df_clean['is_active'] = df_clean['reviews_per_month'] > 0

#### Host Behavior

In [57]:
df_clean['is_professional_host'] = df_clean['calculated_host_listings_count'] > 1

df_clean['host_size'] = pd.cut(df_clean['calculated_host_listings_count'],
                         bins=[0,1,5,100],
                         labels=['single', 'small', 'large'])

#### Localização e Disponibilidade

In [58]:
df_clean['price_vs_neighbourhood'] = df_clean['price'] / df_clean.groupby('neighbourhood')['price'].transform('mean')

df_clean['availability_rate'] = df_clean['availability_365'] / 365

df_clean['availability_level'] = pd.cut(df_clean['availability_365'],
                                  bins=[0,50,200,365],
                                  labels=['rare', 'medium', 'high'])

## 4. Agregação e Métricas

In [59]:
# agg neighbourhood

df_clean.groupby('neighbourhood_group').agg({
    'price' : ['mean', 'median'],
    'number_of_reviews': 'sum',
    'availability_365': 'mean',
    'id': 'count'
})

price        number_of_reviews availability_365  \
                           mean median               sum             mean   
neighbourhood_group                                                         
Bronx                 87.496792   65.0             28371       165.758937   
Brooklyn             124.383207   90.0            486574       100.232292   
Manhattan            196.875814  150.0            454569       111.979410   
Queens                99.517649   75.0            156950       144.451818   
Staten Island        114.812332   75.0             11541       199.678284   

                        id  
                     count  
neighbourhood_group         
Bronx                 1091  
Brooklyn             20104  
Manhattan            21661  
Queens                5666  
Staten Island          373

In [60]:
df_clean.groupby('room_type').agg({
    'price': ['mean', 'median'],
    'availability_365': 'mean',
    'number_of_reviews': 'sum'
})

price        availability_365 number_of_reviews
                       mean median             mean               sum
room_type                                                            
Entire home/apt  211.794246  160.0       111.920304            580403
Private room      89.780973   70.0       111.203933            538346
Shared room       70.127586   45.0       162.000862             19256

In [61]:
df_clean.groupby('neighbourhood').agg({
    'price': 'mean',
    'number_of_reviews': 'sum',
    'id': 'count'
}).sort_values(by='price', ascending=False)

,price,number_of_reviews,id
neighbourhood,,,
Fort Wadsworth,800.000000,0,1
Woodrow,700.000000,0,1
Tribeca,490.638418,2034,177
Sea Gate,487.857143,10,7
Riverdale,442.090909,293,11
...,...,...,...
New Dorp,57.000000,0,1
Soundview,53.466667,441,15
Tremont,51.545455,227,11


In [62]:
df_clean['estimated_revenue'] = df_clean['price'] * df_clean['reviews_per_month'] * 12

df_clean.groupby('neighbourhood_group')['estimated_revenue'].sum()

neighbourhood_group
Bronx             1525518.24
Brooklyn         29976156.60
Manhattan        44885225.28
Queens            9394327.56
Staten Island      592074.96
Name: estimated_revenue, dtype: float64

In [63]:
df_clean['occupancy_estimate'] = df_clean['reviews_per_month'] * 12

In [64]:
df_clean['score'] = (
    df_clean['reviews_per_month'] * 0.5 +
    (1 / df_clean['price']) * 0.3 +
    df_clean['availability_rate'] * 0.2
)

In [65]:
df_clean.head()

,id,name,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,...,demand_level,is_active,is_professional_host,host_size,price_vs_neighbourhood,availability_rate,availability_level,estimated_revenue,occupancy_estimate,score
0,2539,Clean & quiet apt home by the park,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,...,low,True,True,large,1.604122,1.000000,high,375.48,2.52,0.307013
1,2595,Skylit Midtown Castle,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,...,low,True,True,small,0.795843,0.972603,high,1026.00,4.56,0.385854
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,...,NaN,False,False,single,1.260779,1.000000,high,0.00,0.00,0.202000
3,3831,Cozy Entire Floor of Brownstone,4869,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,...,high,True,False,single,0.489298,0.531507,medium,4955.52,55.68,2.429672
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,...,low,True,False,single,0.600606,0.000000,NaN,96.00,1.20,0.053750


## 5. Load - Parquet

In [67]:
output_path = Path('../data/output/AB_NYC_2019.parquet')
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_parquet(output_path, engine='pyarrow', index=False)
size_mb = output_path.stat().st_size / (1024 * 1024)
print(f"=== PARQUET ===")
print(f"Arquivo: {output_path}")
print(f"Tamanho: {size_mb:.1f} MB")

=== PARQUET ===
Arquivo: ..\data\output\AB_NYC_2019.parquet
Tamanho: 3.5 MB


In [68]:
# Validação: ler de volta o Parquet e conferir
df_parquet = pd.read_parquet(output_path)
print(f"=== VALIDAÇÃO ===")
print(f"Registros no Parquet: {len(df_parquet):,}")
print(f"Registros esperados:  {len(df_clean):,}")
print(f"Match: {'OK' if len(df_parquet) == len(df_clean) else 'ERRO'}")
print(f"\nColunas: {list(df_parquet.columns)}")
df_parquet.head()

=== VALIDAÇÃO ===
Registros no Parquet: 48,895
Registros esperados:  48,895
Match: OK

Colunas: ['id', 'name', 'host_id', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365', 'price_per_min_night', 'price_per_review', 'price_category', 'demand_level', 'is_active', 'is_professional_host', 'host_size', 'price_vs_neighbourhood', 'availability_rate', 'availability_level', 'estimated_revenue', 'occupancy_estimate', 'score']


,id,name,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,...,demand_level,is_active,is_professional_host,host_size,price_vs_neighbourhood,availability_rate,availability_level,estimated_revenue,occupancy_estimate,score
0,2539,Clean & quiet apt home by the park,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,...,low,True,True,large,1.604122,1.000000,high,375.48,2.52,0.307013
1,2595,Skylit Midtown Castle,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,...,low,True,True,small,0.795843,0.972603,high,1026.00,4.56,0.385854
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,...,NaN,False,False,single,1.260779,1.000000,high,0.00,0.00,0.202000
3,3831,Cozy Entire Floor of Brownstone,4869,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,...,high,True,False,single,0.489298,0.531507,medium,4955.52,55.68,2.429672
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,...,low,True,False,single,0.600606,0.000000,NaN,96.00,1.20,0.053750
